# Euclidean Similarity Baseline for 15-Coin TDA Benchmark

This notebook adds a simpler Euclidean-distance baseline to the existing 15-coin pipeline.

Workflow:
1. Read the existing 15-coin log-return embeddings.
2. Compute Euclidean pairwise distance matrices for all 12 `(m, tau)` settings.
3. Run the same TDA pipeline to obtain `Betti-0`, `Betti-1`, and `Persistent Entropy`.
4. Merge the Euclidean TDA indicators into the existing multi-window benchmark panel.
5. Recompute ROC-AUC results under the same evaluation windows.
6. Export comparison tables focused on distance-method performance.

All generated data are saved under `8. New data` without overwriting the existing DTW / Wasserstein outputs.


In [1]:
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
from ripser import ripser
from sklearn.metrics import roc_curve, auc

EMBEDDING_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_log_return_embeddings')
SIM_ROOT = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices')
EUC_SIM_DIR = SIM_ROOT / 'euclidean'
EU_TDA_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv_euclidean')
BASE_BENCH_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin')
BENCH_OUT_DIR = Path('/Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin_with_euclidean')

EUC_SIM_DIR.mkdir(parents=True, exist_ok=True)
EU_TDA_DIR.mkdir(parents=True, exist_ok=True)
BENCH_OUT_DIR.mkdir(parents=True, exist_ok=True)

embedding_files = sorted(EMBEDDING_DIR.glob('cmc15_log_return_embedding_m*_tau*.npz'))
print('Embedding files:', len(embedding_files))
print('Euclidean similarity dir:', EUC_SIM_DIR)
print('Euclidean TDA dir:', EU_TDA_DIR)
print('Benchmark output dir:', BENCH_OUT_DIR)


Embedding files: 12
Euclidean similarity dir: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_similarity_matrices/euclidean
Euclidean TDA dir: /Users/jane/Documents/202511吾-Systems/8. New data/15coin_tda_csv_euclidean
Benchmark output dir: /Users/jane/Documents/202511吾-Systems/8. New data/benchmark_multiwindow_15coin_with_euclidean


In [2]:
def compute_euclidean_matrices(panel):
    n_time, n_coins, _ = panel.shape
    out = np.zeros((n_time, n_coins, n_coins), dtype=np.float32)

    for t in range(n_time):
        diff = panel[t][:, None, :] - panel[t][None, :, :]
        out[t] = np.sqrt(np.sum(diff * diff, axis=-1)).astype(np.float32)

    return out


pattern = re.compile(r'm(\d+)_tau(\d+)')


def parse_parameters_from_name(path):
    match = pattern.search(path.stem)
    if not match:
        raise ValueError(f'Could not parse window_size and lag from {path.name}')
    return int(match.group(1)), int(match.group(2))


def save_similarity_npz(output_path, similarity_matrices, coins, dates, window_size, lag, method, value_type='log_return'):
    np.savez_compressed(
        output_path,
        similarity_matrices=similarity_matrices.astype(np.float32),
        coins=np.array(coins, dtype=str),
        dates=np.array(dates, dtype=str),
        window_size=np.int64(window_size),
        lag=np.int64(lag),
        method=np.array(method, dtype=str),
        value_type=np.array(value_type, dtype=str),
        n_time=np.int64(similarity_matrices.shape[0]),
        n_coins=np.int64(similarity_matrices.shape[1]),
    )


warm_panel = np.load(embedding_files[0], allow_pickle=True)['embedded_array'][:2].astype(np.float64)
_ = compute_euclidean_matrices(warm_panel)
print('Euclidean function ready.')


Euclidean function ready.


In [3]:
sim_manifest_rows = []

for embedding_path in embedding_files:
    data = np.load(embedding_path, allow_pickle=True)
    panel = data['embedded_array'].astype(np.float64)
    coins = data['coins'].tolist()
    dates = data['dates'].tolist()
    window_size, lag = parse_parameters_from_name(embedding_path)

    start = time.time()
    euclidean_mats = compute_euclidean_matrices(panel)
    runtime = time.time() - start

    output_path = EUC_SIM_DIR / f'cmc15_log_return_euclidean_similarity_m{window_size:02d}_tau{lag}.npz'
    save_similarity_npz(
        output_path,
        euclidean_mats,
        coins=coins,
        dates=dates,
        window_size=window_size,
        lag=lag,
        method='euclidean',
    )

    sim_manifest_rows.append({
        'embedding_file': embedding_path.name,
        'method': 'euclidean',
        'output_file': output_path.name,
        'window_size': window_size,
        'lag': lag,
        'n_time': euclidean_mats.shape[0],
        'n_coins': euclidean_mats.shape[1],
        'first_date': dates[0],
        'last_date': dates[-1],
        'runtime_seconds': round(runtime, 4),
    })

    print(f'Saved {output_path.name} ({runtime:.2f}s)')

sim_manifest = pd.DataFrame(sim_manifest_rows).sort_values(['window_size', 'lag']).reset_index(drop=True)
sim_manifest.to_csv(EUC_SIM_DIR / 'cmc15_euclidean_similarity_manifest.csv', index=False)
sim_manifest

Saved cmc15_log_return_euclidean_similarity_m07_tau1.npz (0.01s)
Saved cmc15_log_return_euclidean_similarity_m07_tau2.npz (0.01s)
Saved cmc15_log_return_euclidean_similarity_m07_tau3.npz (0.01s)


Saved cmc15_log_return_euclidean_similarity_m14_tau1.npz (0.01s)
Saved cmc15_log_return_euclidean_similarity_m14_tau2.npz (0.01s)


Saved cmc15_log_return_euclidean_similarity_m14_tau3.npz (0.01s)
Saved cmc15_log_return_euclidean_similarity_m21_tau1.npz (0.01s)


Saved cmc15_log_return_euclidean_similarity_m21_tau2.npz (0.01s)
Saved cmc15_log_return_euclidean_similarity_m21_tau3.npz (0.01s)


Saved cmc15_log_return_euclidean_similarity_m28_tau1.npz (0.02s)
Saved cmc15_log_return_euclidean_similarity_m28_tau2.npz (0.03s)


Saved cmc15_log_return_euclidean_similarity_m28_tau3.npz (0.02s)


,embedding_file,method,output_file,window_size,lag,n_time,n_coins,first_date,last_date,runtime_seconds
0,cmc15_log_return_embedding_m07_tau1.npz,euclidean,cmc15_log_return_euclidean_similarity_m07_tau1...,7,1,1867,15,2020-08-17,2025-09-26,0.0110
1,cmc15_log_return_embedding_m07_tau2.npz,euclidean,cmc15_log_return_euclidean_similarity_m07_tau2...,7,2,1861,15,2020-08-23,2025-09-26,0.0119
2,cmc15_log_return_embedding_m07_tau3.npz,euclidean,cmc15_log_return_euclidean_similarity_m07_tau3...,7,3,1855,15,2020-08-29,2025-09-26,0.0116
3,cmc15_log_return_embedding_m14_tau1.npz,euclidean,cmc15_log_return_euclidean_similarity_m14_tau1...,14,1,1860,15,2020-08-24,2025-09-26,0.0121
4,cmc15_log_return_embedding_m14_tau2.npz,euclidean,cmc15_log_return_euclidean_similarity_m14_tau2...,14,2,1847,15,2020-09-06,2025-09-26,0.0121
5,cmc15_log_return_embedding_m14_tau3.npz,euclidean,cmc15_log_return_euclidean_similarity_m14_tau3...,14,3,1834,15,2020-09-19,2025-09-26,0.0122
6,cmc15_log_return_embedding_m21_tau1.npz,euclidean,cmc15_log_return_euclidean_similarity_m21_tau1...,21,1,1853,15,2020-08-31,2025-09-26,0.0147
7,cmc15_log_return_embedding_m21_tau2.npz,euclidean,cmc15_log_return_euclidean_similarity_m21_tau2...,21,2,1833,15,2020-09-20,2025-09-26,0.0148
8,cmc15_log_return_embedding_m21_tau3.npz,euclidean,cmc15_log_return_euclidean_similarity_m21_tau3...,21,3,1813,15,2020-10-10,2025-09-26,0.0141
9,cmc15_log_return_embedding_m28_tau1.npz,euclidean,cmc15_log_return_euclidean_similarity_m28_tau1...,28,1,1846,15,2020-09-07,2025-09-26,0.0168


In [4]:
tda_pattern = re.compile(r'euclidean_similarity_m(\d+)_tau(\d+)')


def parse_euclidean_info(path):
    match = tda_pattern.search(path.stem)
    if not match:
        raise ValueError(f'Cannot parse Euclidean filename: {path.name}')
    return int(match.group(1)), int(match.group(2))


def betti_numbers(diagrams, thresholds):
    betti_0 = np.zeros(len(thresholds), dtype=float)
    betti_1 = np.zeros(len(thresholds), dtype=float)

    if len(diagrams[0]) > 0:
        births_0 = diagrams[0][:, 0]
        deaths_0 = diagrams[0][:, 1]
        for i, eps in enumerate(thresholds):
            betti_0[i] = np.sum((births_0 <= eps) & (deaths_0 > eps))

    if len(diagrams[1]) > 0:
        births_1 = diagrams[1][:, 0]
        deaths_1 = diagrams[1][:, 1]
        for i, eps in enumerate(thresholds):
            betti_1[i] = np.sum((births_1 <= eps) & (deaths_1 > eps))

    return betti_0, betti_1


def persistent_entropy(diagram):
    if len(diagram) == 0:
        return 0.0

    lifetimes = diagram[:, 1] - diagram[:, 0]
    finite = np.isfinite(lifetimes) & (lifetimes > 0)
    lifetimes = lifetimes[finite]

    if len(lifetimes) == 0:
        return 0.0

    probs = lifetimes / lifetimes.sum()
    return float(-(probs * np.log(probs)).sum())


euc_files = sorted(EUC_SIM_DIR.glob('cmc15_log_return_euclidean_similarity_m*_tau*.npz'))
tda_manifest_rows = []

for sim_path in euc_files:
    data = np.load(sim_path, allow_pickle=True)
    similarity_matrices = data['similarity_matrices'].astype(float)
    dates = data['dates'].tolist()
    coins = data['coins'].tolist()
    window_size, lag = parse_euclidean_info(sim_path)

    epsilons = np.linspace(0, np.nanmax(similarity_matrices), 50)
    n_time = similarity_matrices.shape[0]
    betti0_mean = np.zeros(n_time, dtype=float)
    betti1_mean = np.zeros(n_time, dtype=float)
    entropy_series = np.zeros(n_time, dtype=float)

    start = time.time()
    for t in range(n_time):
        dist_mat = similarity_matrices[t]
        dist_mat = np.nan_to_num(dist_mat)
        dist_mat = (dist_mat + dist_mat.T) / 2.0
        np.fill_diagonal(dist_mat, 0.0)

        result = ripser(dist_mat, distance_matrix=True, maxdim=1)
        diagrams = result['dgms']
        b0_curve, b1_curve = betti_numbers(diagrams, epsilons)
        betti0_mean[t] = b0_curve.mean()
        betti1_mean[t] = b1_curve.mean()
        entropy_series[t] = persistent_entropy(diagrams[1])

    runtime = time.time() - start

    df = pd.DataFrame({
        'time_index': np.arange(n_time),
        'date': dates,
        'betti0': betti0_mean,
        'betti1': betti1_mean,
        'persistent_entropy': entropy_series,
        'method': 'euclidean',
        'window_size': window_size,
        'lag': lag,
        'n_coins': len(coins),
        'value_type': 'log_return',
    })

    out_csv = EU_TDA_DIR / f'cmc15_tda_euclidean_m{window_size:02d}_tau{lag}.csv'
    df.to_csv(out_csv, index=False)

    tda_manifest_rows.append({
        'input_file': sim_path.name,
        'output_file': out_csv.name,
        'method': 'euclidean',
        'window_size': window_size,
        'lag': lag,
        'n_time': n_time,
        'n_coins': len(coins),
        'first_date': dates[0],
        'last_date': dates[-1],
        'runtime_seconds': round(runtime, 4),
    })

    print(f'Saved {out_csv.name} ({runtime:.2f}s)')

tda_manifest = pd.DataFrame(tda_manifest_rows).sort_values(['window_size', 'lag']).reset_index(drop=True)
tda_manifest.to_csv(EU_TDA_DIR / 'cmc15_tda_euclidean_manifest.csv', index=False)
tda_manifest

Saved cmc15_tda_euclidean_m07_tau1.csv (0.38s)


Saved cmc15_tda_euclidean_m07_tau2.csv (0.38s)


Saved cmc15_tda_euclidean_m07_tau3.csv (0.39s)


Saved cmc15_tda_euclidean_m14_tau1.csv (0.36s)


Saved cmc15_tda_euclidean_m14_tau2.csv (0.36s)


...

Saved cmc15_tda_euclidean_m21_tau3.csv (0.34s)


Saved cmc15_tda_euclidean_m28_tau1.csv (0.34s)


Saved cmc15_tda_euclidean_m28_tau2.csv (0.33s)


Saved cmc15_tda_euclidean_m28_tau3.csv (0.34s)


,input_file,output_file,method,window_size,lag,n_time,n_coins,first_date,last_date,runtime_seconds
0,cmc15_log_return_euclidean_similarity_m07_tau1...,cmc15_tda_euclidean_m07_tau1.csv,euclidean,7,1,1867,15,2020-08-17,2025-09-26,0.3785
1,cmc15_log_return_euclidean_similarity_m07_tau2...,cmc15_tda_euclidean_m07_tau2.csv,euclidean,7,2,1861,15,2020-08-23,2025-09-26,0.3758
2,cmc15_log_return_euclidean_similarity_m07_tau3...,cmc15_tda_euclidean_m07_tau3.csv,euclidean,7,3,1855,15,2020-08-29,2025-09-26,0.3861
3,cmc15_log_return_euclidean_similarity_m14_tau1...,cmc15_tda_euclidean_m14_tau1.csv,euclidean,14,1,1860,15,2020-08-24,2025-09-26,0.3600
4,cmc15_log_return_euclidean_similarity_m14_tau2...,cmc15_tda_euclidean_m14_tau2.csv,euclidean,14,2,1847,15,2020-09-06,2025-09-26,0.3624
5,cmc15_log_return_euclidean_similarity_m14_tau3...,cmc15_tda_euclidean_m14_tau3.csv,euclidean,14,3,1834,15,2020-09-19,2025-09-26,0.3659
6,cmc15_log_return_euclidean_similarity_m21_tau1...,cmc15_tda_euclidean_m21_tau1.csv,euclidean,21,1,1853,15,2020-08-31,2025-09-26,0.3467
7,cmc15_log_return_euclidean_similarity_m21_tau2...,cmc15_tda_euclidean_m21_tau2.csv,euclidean,21,2,1833,15,2020-09-20,2025-09-26,0.3426
8,cmc15_log_return_euclidean_similarity_m21_tau3...,cmc15_tda_euclidean_m21_tau3.csv,euclidean,21,3,1813,15,2020-10-10,2025-09-26,0.3446
9,cmc15_log_return_euclidean_similarity_m28_tau1...,cmc15_tda_euclidean_m28_tau1.csv,euclidean,28,1,1846,15,2020-09-07,2025-09-26,0.3359


In [5]:
base_panel = pd.read_csv(BASE_BENCH_DIR / 'cmc15_benchmark_panel_with_labels.csv', parse_dates=['date'])
base_manifest = pd.read_csv(BASE_BENCH_DIR / 'cmc15_indicator_manifest.csv')
eval_windows = pd.read_csv(BASE_BENCH_DIR / 'cmc15_evaluation_windows.csv')

pattern = re.compile(r'cmc15_tda_euclidean_m(\d+)_tau(\d+)\.csv')

euc_tda_files = sorted(EU_TDA_DIR.glob('cmc15_tda_euclidean_m*_tau*.csv'))

euc_wide = None
euc_manifest_rows = []

for path in euc_tda_files:
    match = pattern.match(path.name)
    if not match:
        raise ValueError(f'Unexpected Euclidean TDA filename: {path.name}')
    window_size = int(match.group(1))
    lag = int(match.group(2))

    df = pd.read_csv(path, parse_dates=['date'])
    prefix = f'tda_euclidean_m{window_size:02d}_tau{lag}'

    rename_map = {
        'betti0': f'{prefix}_betti0',
        'betti1': f'{prefix}_betti1',
        'persistent_entropy': f'{prefix}_entropy',
    }

    part = df[['date', 'betti0', 'betti1', 'persistent_entropy']].rename(columns=rename_map)

    if euc_wide is None:
        euc_wide = part.copy()
    else:
        euc_wide = euc_wide.merge(part, on='date', how='outer')

    for feature_name, indicator_col in [('betti0', rename_map['betti0']), ('betti1', rename_map['betti1']), ('entropy', rename_map['persistent_entropy'])]:
        euc_manifest_rows.append({
            'indicator': indicator_col,
            'category': 'tda',
            'distance_method': 'euclidean',
            'embedding_window_size': window_size,
            'embedding_lag': lag,
            'feature': feature_name,
            'source_file': path.name,
        })

euc_wide = euc_wide.sort_values('date').reset_index(drop=True)
euc_wide.to_csv(BENCH_OUT_DIR / 'cmc15_euclidean_tda_wide_panel.csv', index=False)

combined_panel = base_panel.merge(euc_wide, on='date', how='left')
combined_panel.to_csv(BENCH_OUT_DIR / 'cmc15_benchmark_panel_with_euclidean.csv', index=False)

combined_manifest = pd.concat([base_manifest, pd.DataFrame(euc_manifest_rows)], ignore_index=True)
combined_manifest.to_csv(BENCH_OUT_DIR / 'cmc15_indicator_manifest_with_euclidean.csv', index=False)

print('Base panel shape:', base_panel.shape)
print('Combined panel shape:', combined_panel.shape)
print('Combined indicators:', len(combined_manifest))
combined_manifest.tail(12)

Base panel shape: (1867, 87)
Combined panel shape: (1867, 123)
Combined indicators: 115


,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,source_file
103,tda_euclidean_m21_tau3_betti0,tda,euclidean,21.0,3.0,betti0,cmc15_tda_euclidean_m21_tau3.csv
104,tda_euclidean_m21_tau3_betti1,tda,euclidean,21.0,3.0,betti1,cmc15_tda_euclidean_m21_tau3.csv
105,tda_euclidean_m21_tau3_entropy,tda,euclidean,21.0,3.0,entropy,cmc15_tda_euclidean_m21_tau3.csv
106,tda_euclidean_m28_tau1_betti0,tda,euclidean,28.0,1.0,betti0,cmc15_tda_euclidean_m28_tau1.csv
107,tda_euclidean_m28_tau1_betti1,tda,euclidean,28.0,1.0,betti1,cmc15_tda_euclidean_m28_tau1.csv
108,tda_euclidean_m28_tau1_entropy,tda,euclidean,28.0,1.0,entropy,cmc15_tda_euclidean_m28_tau1.csv
109,tda_euclidean_m28_tau2_betti0,tda,euclidean,28.0,2.0,betti0,cmc15_tda_euclidean_m28_tau2.csv
110,tda_euclidean_m28_tau2_betti1,tda,euclidean,28.0,2.0,betti1,cmc15_tda_euclidean_m28_tau2.csv
111,tda_euclidean_m28_tau2_entropy,tda,euclidean,28.0,2.0,entropy,cmc15_tda_euclidean_m28_tau2.csv
112,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,cmc15_tda_euclidean_m28_tau3.csv


In [6]:
label_cols = [c for c in combined_panel.columns if c.startswith('event_label_')]
all_auc_indicators = combined_manifest['indicator'].tolist()
meta_map = combined_manifest.drop_duplicates('indicator').set_index('indicator').to_dict('index')


def evaluable_event_count(series_dates, events, start_offset, end_offset):
    series_start = pd.to_datetime(series_dates.min())
    series_end = pd.to_datetime(series_dates.max())
    count = 0
    for e in events:
        ws = e + pd.Timedelta(days=start_offset)
        we = e + pd.Timedelta(days=end_offset)
        if ws >= series_start and we <= series_end:
            count += 1
    return count


events = pd.to_datetime([
    '2020-12-01','2021-01-02','2021-01-07','2021-01-29','2021-02-16','2021-03-13',
    '2021-04-10','2021-05-12','2021-05-17','2021-05-18','2021-05-19','2021-06-09',
    '2021-09-24','2021-10-15','2021-11-15','2022-04-27','2022-05-01','2022-05-11',
    '2022-05-12','2022-05-13','2022-07-20','2022-11-01','2023-03-01','2023-03-10',
    '2023-05-17','2023-06-16','2023-07-01','2023-10-01','2024-03-19','2024-04-20',
    '2025-01-20','2025-02-03','2025-02-21','2025-03-07','2025-05-20','2025-06-05','2025-06-17'
])

auc_rows = []
for _, w in eval_windows.iterrows():
    window_label = w['window_label']
    start_offset = int(w['start_offset'])
    end_offset = int(w['end_offset'])
    label_col = f'event_label_{window_label}'
    y_full = combined_panel[label_col].to_numpy()

    for indicator in all_auc_indicators:
        y_score = pd.to_numeric(combined_panel[indicator], errors='coerce').to_numpy()
        mask = np.isfinite(y_score)

        if mask.sum() == 0:
            continue

        y_true = y_full[mask]
        y_score_masked = y_score[mask]
        if np.unique(y_true).size < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true, y_score_masked)
        roc_auc = auc(fpr, tpr)
        series_dates = combined_panel.loc[mask, 'date']
        meta = meta_map[indicator]

        auc_rows.append({
            'window_label': window_label,
            'start_offset': start_offset,
            'end_offset': end_offset,
            'indicator': indicator,
            'category': meta['category'],
            'distance_method': meta['distance_method'],
            'embedding_window_size': meta['embedding_window_size'],
            'embedding_lag': meta['embedding_lag'],
            'feature': meta['feature'],
            'auc': roc_auc,
            'n_obs': int(mask.sum()),
            'n_positive': int(y_true.sum()),
            'n_events_in_range': int(evaluable_event_count(series_dates, events, start_offset, end_offset)),
        })

auc_df = pd.DataFrame(auc_rows).sort_values(['window_label', 'auc'], ascending=[True, False]).reset_index(drop=True)
auc_df.to_csv(BENCH_OUT_DIR / 'cmc15_auc_multiwindow_long.csv', index=False)

indicator_meta = combined_manifest.drop_duplicates('indicator').copy()
auc_mean = auc_df.groupby('indicator', as_index=False)['auc'].mean().rename(columns={'auc': 'mean_auc'})
auc_wide = auc_df.pivot(index='indicator', columns='window_label', values='auc').reset_index()
auc_summary = indicator_meta.merge(auc_mean, on='indicator', how='left').merge(auc_wide, on='indicator', how='left')
auc_summary = auc_summary.sort_values('mean_auc', ascending=False).reset_index(drop=True)
auc_summary.to_csv(BENCH_OUT_DIR / 'cmc15_auc_indicator_summary.csv', index=False)

auc_best = auc_df.sort_values(['window_label', 'auc'], ascending=[True, False]).drop_duplicates('window_label').reset_index(drop=True)
auc_best.to_csv(BENCH_OUT_DIR / 'cmc15_auc_best_by_window.csv', index=False)

best_tda_by_distance = (
    auc_df.loc[auc_df['category'] == 'tda']
    .sort_values(['window_label', 'distance_method', 'auc'], ascending=[True, True, False])
    .drop_duplicates(['window_label', 'distance_method'])
    .reset_index(drop=True)
)
best_tda_by_distance.to_csv(BENCH_OUT_DIR / 'cmc15_auc_best_tda_by_distance_window.csv', index=False)

summary_tda = auc_summary.loc[auc_summary['category'] == 'tda'].copy()
distance_best = summary_tda.sort_values(['distance_method', 'mean_auc'], ascending=[True, False]).drop_duplicates('distance_method')
distance_agg = (
    summary_tda.groupby('distance_method', as_index=False)
    .agg(
        n_indicators=('indicator', 'count'),
        mean_of_mean_auc=('mean_auc', 'mean'),
        median_of_mean_auc=('mean_auc', 'median'),
        max_mean_auc=('mean_auc', 'max'),
    )
    .merge(distance_best[['distance_method', 'indicator', 'mean_auc']], on='distance_method', how='left', suffixes=('', '_best_indicator'))
    .rename(columns={'indicator': 'best_indicator', 'mean_auc': 'best_indicator_mean_auc'})
    .sort_values('mean_of_mean_auc', ascending=False)
    .reset_index(drop=True)
)
distance_agg.to_csv(BENCH_OUT_DIR / 'cmc15_auc_distance_method_summary.csv', index=False)

print('AUC rows:', len(auc_df))
auc_df.head(10)

AUC rows: 575


,window_label,start_offset,end_offset,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,auc,n_obs,n_positive,n_events_in_range
0,"[-14,0]",-14,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.636218,1867,452,37
1,"[-14,0]",-14,0,tda_dtw_m28_tau3_betti0,tda,dtw,28.0,3.0,betti0,0.634017,1792,452,37
2,"[-14,0]",-14,0,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,0.633098,1792,452,37
3,"[-14,0]",-14,0,tda_euclidean_m07_tau1_betti0,tda,euclidean,7.0,1.0,betti0,0.631804,1867,452,37
4,"[-14,0]",-14,0,tda_euclidean_m14_tau1_betti0,tda,euclidean,14.0,1.0,betti0,0.629819,1860,452,37
5,"[-14,0]",-14,0,tda_dtw_m14_tau1_betti0,tda,dtw,14.0,1.0,betti0,0.629199,1860,452,37
6,"[-14,0]",-14,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.628166,1867,452,37
7,"[-14,0]",-14,0,tda_wasserstein_m28_tau3_betti0,tda,wasserstein,28.0,3.0,betti0,0.624743,1792,452,37
8,"[-14,0]",-14,0,CVI,benchmark,NaN,NaN,NaN,CVI,0.622806,1564,352,30
9,"[-14,0]",-14,0,tda_dtw_m07_tau2_betti0,tda,dtw,7.0,2.0,betti0,0.622726,1861,452,37


In [7]:
print('Saved Euclidean benchmark files:')
for p in sorted(BENCH_OUT_DIR.glob('*')):
    print('-', p.name)

print()
print('Top Euclidean / DTW / Wasserstein TDA rows by window:')
display(pd.read_csv(BENCH_OUT_DIR / 'cmc15_auc_best_tda_by_distance_window.csv').head(15))

print()
print('Distance-method summary:')
display(pd.read_csv(BENCH_OUT_DIR / 'cmc15_auc_distance_method_summary.csv'))


Saved Euclidean benchmark files:
- cmc15_auc_best_by_window.csv
- cmc15_auc_best_tda_by_distance_window.csv
- cmc15_auc_distance_method_summary.csv
- cmc15_auc_indicator_summary.csv
- cmc15_auc_multiwindow_long.csv
- cmc15_benchmark_panel_with_euclidean.csv
- cmc15_euclidean_tda_wide_panel.csv
- cmc15_indicator_manifest_with_euclidean.csv

Top Euclidean / DTW / Wasserstein TDA rows by window:


,window_label,start_offset,end_offset,indicator,category,distance_method,embedding_window_size,embedding_lag,feature,auc,n_obs,n_positive,n_events_in_range
0,"[-14,0]",-14,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.636218,1867,452,37
1,"[-14,0]",-14,0,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,0.633098,1792,452,37
2,"[-14,0]",-14,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.628166,1867,452,37
3,"[-21,0]",-21,0,tda_dtw_m28_tau3_betti0,tda,dtw,28.0,3.0,betti0,0.662981,1792,604,37
4,"[-21,0]",-21,0,tda_euclidean_m28_tau3_betti0,tda,euclidean,28.0,3.0,betti0,0.664299,1792,604,37
5,"[-21,0]",-21,0,tda_wasserstein_m28_tau3_betti0,tda,wasserstein,28.0,3.0,betti0,0.649236,1792,604,37
6,"[-7,0]",-7,0,tda_dtw_m07_tau1_betti0,tda,dtw,7.0,1.0,betti0,0.646456,1867,258,37
7,"[-7,0]",-7,0,tda_euclidean_m07_tau1_betti0,tda,euclidean,7.0,1.0,betti0,0.640397,1867,258,37
8,"[-7,0]",-7,0,tda_wasserstein_m07_tau1_betti0,tda,wasserstein,7.0,1.0,betti0,0.649221,1867,258,37
9,"[0,+21]",0,21,tda_dtw_m28_tau1_betti0,tda,dtw,28.0,1.0,betti0,0.687707,1846,604,37



Distance-method summary:


,distance_method,n_indicators,mean_of_mean_auc,median_of_mean_auc,max_mean_auc,best_indicator,best_indicator_mean_auc
0,dtw,36,0.553911,0.527745,0.644919,tda_dtw_m14_tau1_betti0,0.644919
1,euclidean,36,0.552859,0.525112,0.641784,tda_euclidean_m14_tau1_betti0,0.641784
2,wasserstein,36,0.549837,0.524419,0.638191,tda_wasserstein_m07_tau1_betti0,0.638191
